# GuFu chain diagnostics

This notebook compares the three links in the saved training runs. It evaluates both the online updater that produced round 1 and the post-transition updater that produced round 2, for both experiment directories.

| Link | Comparison |
|---|---|
| GNN embedding → position | Saved updater predictions versus a fresh reference-trained probe, both scored on held-out update locations |
| Position → AE latent | `F(predicted position)` and `F(true position)` versus the saved update AE encoder target |
| AE latent → RSS | Decode those two latents and the AE target; score only cells physically measured in each update scan |

The true update coordinates are used only for evaluation and the oracle-position comparison. They are not used to train either updater or the fresh probe. RSS decoder errors are in dBm and exclude unmeasured cells. The AE-target decode is a reference point for the decoder, not a ground-truth location-preserving latent.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from gufu_debug_evaluation import evaluate_debug_run

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'gufu.py').is_file():
    raise RuntimeError('Run this notebook with the GUFU repository root as the working directory.')

OUTPUT_ROOTS = {
    'GF_from1F_12': PROJECT_ROOT / 'output_from1F_12',
    'GF_fromGF_12': PROJECT_ROOT / 'output_fromGF_12',
}
DEVICE_OVERRIDE = None  # None uses each saved run's device.

## Run the diagnostics on saved artifacts

This reads saved GNN embeddings, updater checkpoints, encoder targets, and AE checkpoints. It does not rerun the streaming pipeline or train the GNN/AE. It does fit the small reference-only localization probes used for the first comparison.

In [2]:
summary_frames = []
per_scan_frames = []
for run_name, output_root in OUTPUT_ROOTS.items():
    if not output_root.is_dir():
        raise FileNotFoundError(f"Saved output directory not found: {output_root}")
    print(f"Auditing {run_name} from {output_root}...")
    run_summary, run_per_scan = evaluate_debug_run(
        run_name, output_root, device_override=DEVICE_OVERRIDE
    )
    summary_frames.append(run_summary)
    per_scan_frames.append(run_per_scan)

summary = pd.concat(summary_frames, ignore_index=True)
per_scan = pd.concat(per_scan_frames, ignore_index=True)
summary

Auditing GF_from1F_12 from /home/oem/GUFU/output_from1F_12...
Auditing GF_fromGF_12 from /home/oem/GUFU/output_fromGF_12...


,run,updater_round,held_out_scans,mean_measured_AP_coverage_percent,updater_location_RMSE,fresh_probe_location_RMSE,mean_latent_predicted_position_to_AE_RMSE_per_dim,mean_latent_true_position_to_AE_RMSE_per_dim,mean_latent_shift_from_location_prediction_RMSE_per_dim,decoded_RSS_predicted_position_RMSE_dBm,decoded_RSS_predicted_position_MAE_dBm,decoded_RSS_true_position_RMSE_dBm,decoded_RSS_true_position_MAE_dBm,decoded_RSS_AE_target_RMSE_dBm,decoded_RSS_AE_target_MAE_dBm
0,GF_from1F_12,online_updater_round1_fingerprints,419,12.867628,46.721396,44.630586,0.071971,0.081693,0.036859,11.477726,7.871135,10.684074,7.048191,11.360338,8.115312
1,GF_from1F_12,post_transition_updater_round2_fingerprints,419,12.867628,47.715915,54.127990,0.030150,0.035380,0.022801,10.164043,6.690910,8.934720,5.541044,7.949612,5.066657
2,GF_fromGF_12,online_updater_round1_fingerprints,1733,3.729935,107.530786,148.706869,0.304533,0.337071,0.153151,26.320526,21.297280,24.202702,18.823200,19.853256,14.556502
3,GF_fromGF_12,post_transition_updater_round2_fingerprints,1733,3.729935,64.810210,100.928537,0.136224,0.146281,0.084081,23.571615,18.274922,22.217962,16.507438,18.562258,12.981149


## Find where errors grow

Positive gaps mean the full predicted-position path is worse than the corresponding oracle comparison. The per-scan rows help distinguish location error propagation from a position-to-feature or decoder gap.

In [3]:
per_scan['location_error_gap_actual_minus_probe'] = (
    per_scan['updater_location_error'] - per_scan['fresh_probe_location_error']
)
per_scan['decoder_RSS_gap_predicted_minus_true_position_dBm'] = (
    per_scan['decoded_RSS_predicted_position_RMSE_dBm']
    - per_scan['decoded_RSS_true_position_RMSE_dBm']
)
per_scan['decoder_RSS_gap_true_position_minus_AE_target_dBm'] = (
    per_scan['decoded_RSS_true_position_RMSE_dBm']
    - per_scan['decoded_RSS_AE_target_RMSE_dBm']
)

cols = ['run', 'updater_round', 'scan_id', 'updater_location_error',
        'fresh_probe_location_error', 'location_error_gap_actual_minus_probe',
        'latent_predicted_position_to_AE_RMSE_per_dim',
        'latent_true_position_to_AE_RMSE_per_dim',
        'decoded_RSS_predicted_position_RMSE_dBm',
        'decoded_RSS_true_position_RMSE_dBm',
        'decoded_RSS_AE_target_RMSE_dBm']
worst_location_gap = (
    per_scan.sort_values('location_error_gap_actual_minus_probe', ascending=False)
    .groupby(['run', 'updater_round'], as_index=False).head(10)
)
display(worst_location_gap[cols])

worst_decoder_gap = (
    per_scan.sort_values('decoder_RSS_gap_predicted_minus_true_position_dBm', ascending=False)
    .groupby(['run', 'updater_round'], as_index=False).head(10)
)
display(worst_decoder_gap[cols])

,run,updater_round,scan_id,updater_location_error,fresh_probe_location_error,location_error_gap_actual_minus_probe,latent_predicted_position_to_AE_RMSE_per_dim,latent_true_position_to_AE_RMSE_per_dim,decoded_RSS_predicted_position_RMSE_dBm,decoded_RSS_true_position_RMSE_dBm,decoded_RSS_AE_target_RMSE_dBm
1298,GF_fromGF_12,online_updater_round1_fingerprints,scan:700627ddd38b4f73ac84d383c703a0ff8e4f496f9...,221.273110,42.924302,178.348808,0.387589,0.521590,24.192990,25.553889,30.690167
1006,GF_fromGF_12,online_updater_round1_fingerprints,scan:9415fe1baa9604c6751037186c6707e3888a875e0...,219.931276,54.003517,165.927759,0.525570,0.557149,25.476917,25.700876,25.655921
1007,GF_fromGF_12,online_updater_round1_fingerprints,scan:c9d04e8a6e225e4a9f2b8a0a7ba9eb2d6901d1d5d...,205.022691,57.561527,147.461164,0.561091,0.549065,25.769767,25.161513,24.603758
1309,GF_fromGF_12,online_updater_round1_fingerprints,scan:74e6b59f0a55d75a26d09934a6afd0d0a4c63d60d...,176.201801,30.117854,146.083948,0.383763,0.620655,27.886278,28.700889,28.202155
913,GF_fromGF_12,online_updater_round1_fingerprints,scan:4ed83a24665e69162b0eb18289c075435a67357e7...,185.487390,45.937647,139.549743,0.682287,0.673846,29.847851,22.791351,29.115874
1308,GF_fromGF_12,online_updater_round1_fingerprints,scan:bb4d08971e124a25cba91c9437c1a6cda8422026c...,169.315547,30.026879,139.288668,0.325688,0.497058,27.710074,27.333836,27.658592
1312,GF_fromGF_12,online_updater_round1_fingerprints,scan:c9bc40bac2367ac3a9ca29e891089970f369dbbe3...,159.680730,22.279517,137.401213,0.297066,0.463703,29.168647,29.581571,27.730907
1311,GF_fromGF_12,online_updater_round1_fingerprints,scan:027d7084985f4d8a4d9941a842d4e92095d9b4d38...,160.363754,23.202556,137.161198,0.332898,0.409324,29.951490,30.197934,28.745424
917,GF_fromGF_12,online_updater_round1_fingerprints,scan:73f5d7327f04fd297830727bde56d6bd40b52d103...,172.662911,35.855908,136.807003,0.749107,0.769661,26.913453,22.853378,32.229219
1319,GF_fromGF_12,online_updater_round1_fingerprints,scan:8da616edd1edd346ba547e758dd1e7e41505530eb...,144.098409,7.687674,136.410736,0.313765,0.415616,30.593619,30.379020,29.944125


,run,updater_round,scan_id,updater_location_error,fresh_probe_location_error,location_error_gap_actual_minus_probe,latent_predicted_position_to_AE_RMSE_per_dim,latent_true_position_to_AE_RMSE_per_dim,decoded_RSS_predicted_position_RMSE_dBm,decoded_RSS_true_position_RMSE_dBm,decoded_RSS_AE_target_RMSE_dBm
2281,GF_fromGF_12,online_updater_round1_fingerprints,scan:4ae5817d30e33a51242d1731e85273579cedbefd8...,272.090330,259.982373,12.107957,0.394584,0.397812,49.036470,12.577921,13.981862
3959,GF_fromGF_12,post_transition_updater_round2_fingerprints,scan:aa50c7b35c6b16663071a23faee14a34b7f75f210...,188.222362,150.467693,37.754669,0.285340,0.286085,64.422747,28.640747,46.615349
2256,GF_fromGF_12,online_updater_round1_fingerprints,scan:365614ea1b0e2c44eb5d3b7056fb099981f2b3d2a...,252.899520,228.231604,24.667915,0.363012,0.404603,48.524659,13.654656,10.583276
2241,GF_fromGF_12,online_updater_round1_fingerprints,scan:fc686cdd0c4dd3b228a85124900e243b0d51dcc3b...,266.833551,233.560119,33.273432,0.369593,0.364353,47.322818,15.043814,15.098873
2249,GF_fromGF_12,online_updater_round1_fingerprints,scan:8196126563906e1635299c56bbca06883ac82ef87...,273.656812,239.435658,34.221154,0.384418,0.380872,47.783339,16.516970,15.132837
4002,GF_fromGF_12,post_transition_updater_round2_fingerprints,scan:22352fb9484603448e71412b4816dc2402cc3e030...,197.837210,138.929687,58.907523,0.204202,0.141478,38.922326,7.711345,11.281988
1536,GF_fromGF_12,online_updater_round1_fingerprints,scan:bfd12172cf1659d2b0cb54fef8c69a71269fcc55c...,309.537941,308.007414,1.530527,0.360712,0.313410,42.202289,12.032361,11.036794
1509,GF_fromGF_12,online_updater_round1_fingerprints,scan:97739b34d8f44c25df44d17abb45a1b5eec790e13...,259.733523,224.776933,34.956590,0.342763,0.366439,45.388299,15.584290,16.015035
2240,GF_fromGF_12,online_updater_round1_fingerprints,scan:ced25e46602567149398b5ad9dfa61f42efffcfb9...,248.875132,219.179684,29.695448,0.383755,0.340871,44.840217,15.217443,18.820298
2252,GF_fromGF_12,online_updater_round1_fingerprints,scan:0cc121a77cb1ad750203d6c8f95127b355ce0bbff...,257.919565,230.174827,27.744738,0.368596,0.380862,44.354331,15.056211,18.169289
